# HARMONY Model - Google Colab Training & Testing

This notebook provides two modes for working with the HARMONY cognitive architecture:

## 🚀 Quick Testing (Inference Only)
- **Time**: 5-10 minutes
- **GPU**: Minimal usage
- **Purpose**: Test pre-trained model, run inference examples
- **Best for**: Quick demos, understanding the model

## 🎯 Full Training Pipeline
- **Time**: 2-3 hours (can be split across sessions)
- **GPU**: Full T4 GPU usage
- **Purpose**: Train model from scratch with all 5 stages
- **Best for**: Training custom models, research

---

### Instructions:
1. **First time setup**: Run the "Setup & Installation" section below
2. **Choose your mode**:
   - Skip to "🚀 Quick Testing" for inference
   - Skip to "🎯 Full Training" for training
3. **Save your work**: Models are saved to Google Drive automatically

### Tips for Free Colab:
- Sessions timeout after 12 hours - use checkpoints to resume
- GPU may not always be available - you can run on CPU (slower)
- Save checkpoints to Google Drive to persist across sessions
- Monitor GPU memory to avoid OOM errors

## 📋 Setup & Installation

Run this section first to set up the environment.

In [ ]:
# Check GPU availability
import torch
print("Checking GPU availability...")
if torch.cuda.is_available():
    print(f"✓ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("✗ No GPU available - will use CPU (slower)")

In [ ]:
# Clone HARMONY repository
!git clone https://github.com/yourusername/harmony-model.git /content/harmony-model
%cd /content/harmony-model/harmony-core
print("✓ Repository cloned")

In [ ]:
# Install dependencies
print("Installing dependencies (this may take 5-10 minutes)...")
!pip install -q torch>=2.0.0 transformers>=4.30.0 tokenizers>=0.13.3 accelerate>=0.20.0
!pip install -q faiss-cpu sentence-transformers datasets pydantic pydantic-settings
!pip install -q tqdm networkx pypdf python-docx
print("✓ All dependencies installed")

In [ ]:
# Install HARMONY in development mode
!pip install -e . -q
print("✓ HARMONY installed")

In [ ]:
# Copy Colab utilities
!cp /content/harmony-model/harmony-colab/colab_utils.py /content/
!cp /content/harmony-model/harmony-colab/config_colab.yaml /content/
print("✓ Colab utilities copied")

In [ ]:
# Mount Google Drive (for saving models)
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
# Import utilities and verify setup
import sys
sys.path.insert(0, '/content')
from colab_utils import ColabManager, print_section_header, check_dependencies

# Verify all dependencies
if check_dependencies():
    print("\n✓ Setup complete! You can now proceed to testing or training.")
else:
    print("\n✗ Setup incomplete. Please check error messages above.")

---

## 🚀 Quick Testing (Inference Only)

This section loads a pre-trained model and runs inference examples.
**Estimated time**: 5-10 minutes

In [ ]:
# Initialize Colab manager
colab_manager = ColabManager(drive_path="/content/drive/MyDrive/HARMONY")
colab_manager.print_memory_status()

In [ ]:
# Download pre-trained model (if available)
# NOTE: Replace with actual model URL when available
import os
model_path = "/content/harmony_pretrained.pt"

# Check if model exists in Drive
if colab_manager.download_from_drive("harmony_pretrained.pt", "/content"):
    print("✓ Model downloaded from Google Drive")
else:
    print("⚠ No pre-trained model found in Drive")
    print("You can either:")
    print("  1. Upload a model to Google Drive and retry")
    print("  2. Skip to the Full Training section to train from scratch")
    print("  3. Continue with untrained model (for testing architecture only)")

In [ ]:
# Load HARMONY model
from harmony.models.harmony_model import HarmonyModel

print("Loading HARMONY model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HarmonyModel().to(device)

# Load pre-trained weights if available
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Pre-trained weights loaded from {model_path}")
else:
    print("⚠ Using untrained model (random weights)")

print(f"✓ Model loaded on {device}")

In [ ]:
# Test inference with a simple query
print_section_header("Testing Inference")

test_query = "What is the capital of France?"
print(f"Query: {test_query}\n")

# Run inference
with torch.no_grad():
    response = model.generate(
        test_query,
        max_length=100,
        temperature=0.7,
        do_sample=True
    )

print(f"Response: {response}")

In [ ]:
# Test with multiple queries
test_queries = [
    "Explain the concept of machine learning.",
    "What are the primary colors?",
    "How does photosynthesis work?",
]

print_section_header("Multiple Inference Tests")

for i, query in enumerate(test_queries, 1):
    print(f"\n--- Query {i} ---")
    print(f"Query: {query}")
    
    with torch.no_grad():
        response = model.generate(
            query,
            max_length=150,
            temperature=0.7,
            do_sample=True
        )
    
    print(f"Response: {response}")

In [ ]:
# Test planner component
print_section_header("Testing Planner Component")

planning_query = "Plan a trip to Paris including flights, accommodation, and sightseeing."
print(f"Planning Query: {planning_query}\n")

# Get planning actions
with torch.no_grad():
    plan = model.planner.generate_plan(planning_query)
    
print("Generated Plan:")
for step, action in enumerate(plan, 1):
    print(f"  {step}. {action}")

In [ ]:
# Test retrieval memory
print_section_header("Testing Retrieval Memory")

# Add some test documents to memory
test_docs = [
    "Paris is the capital of France and known for the Eiffel Tower.",
    "Machine learning is a subset of artificial intelligence.",
    "Photosynthesis converts light energy into chemical energy in plants.",
]

for i, doc in enumerate(test_docs):
    emb = model.embedding_manager.embed_query(doc)
    model.memory_manager.semantic.add_document(f"doc_{i}", doc, emb, {})

print(f"✓ Added {len(test_docs)} documents to memory")

# Test retrieval
query = "What is Paris known for?"
results = model.memory_manager.semantic.search(query, k=2)

print(f"\nQuery: {query}")
print("Retrieved documents:")
for i, (doc_id, score, text) in enumerate(results, 1):
    print(f"  {i}. [{doc_id}] (score: {score:.3f}): {text}")

In [ ]:
# Final memory status
colab_manager.print_memory_status()
print("\n✓ Quick testing complete!")

---

## 🎯 Full Training Pipeline

This section trains the HARMONY model from scratch with all 5 stages.
**Estimated time**: 2-3 hours (can be split across sessions)

### Training Stages:
1. **Stage 1**: Backbone Pretraining (WikiText-103)
2. **Stage 2**: Retrieval Memory Setup
3. **Stage 3**: Planner Training (GSM8K)
4. **Stage 4**: Verifier Training (GSM8K)
5. **Stage 5**: Joint Fine-Tuning

In [ ]:
# Initialize training environment
print_section_header("Initializing Training Environment")

import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import yaml

# Load Colab configuration
with open('/content/config_colab.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training device: {device}")

# Enable mixed precision for GPU
use_amp = device == "cuda" and config['training']['mixed_precision']
if use_amp:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    print("✓ Mixed precision (FP16) enabled")

# Initialize Colab manager
colab_manager = ColabManager(drive_path="/content/drive/MyDrive/HARMONY")
colab_manager.print_memory_status()

In [ ]:
# Download datasets
print_section_header("Downloading Datasets")

# WikiText-103 subset
print("Downloading WikiText-103...")
wikitext = load_dataset('wikitext', 'wikitext-103-v1', split='train')
shard = config['datasets']['wikitext']['subset_shard']
wikitext = wikitext.shard(num_shards=shard, index=0)
print(f"✓ WikiText subset: {len(wikitext)} samples")

# GSM8K for planner/verifier
print("\nDownloading GSM8K...")
gsm8k = load_dataset('gsm8k', 'main', split='train')
gsm8k_samples = config['datasets']['gsm8k']['samples']
gsm8k = gsm8k.select(range(min(len(gsm8k), gsm8k_samples)))
print(f"✓ GSM8K: {len(gsm8k)} samples")

print("\n✓ All datasets downloaded")

In [ ]:
# Initialize HARMONY model
print_section_header("Initializing HARMONY Model")

from harmony.models.harmony_model import HarmonyModel
from harmony.training.orchestrator import TrainingOrchestrator
from harmony.data.dataset_manager import DatasetManager
from harmony.ingestion.pipeline import KnowledgeIngestionPipeline
from harmony.planner.training import PlannerDataset, PlannerTrainer
from harmony.verifier.training import VerifierDataset, VerifierTrainer

# Create model with Colab config
model_config = config['model']
print(f"Model config: {model_config['hidden_size']} hidden, {model_config['num_layers']} layers")

model = HarmonyModel(
    hidden_size=model_config['hidden_size'],
    num_layers=model_config['num_layers'],
    num_attention_heads=model_config['num_attention_heads'],
    max_position_embeddings=model_config['max_position_embeddings']
).to(device)

# Enable gradient checkpointing for memory efficiency
if config['memory']['enable_gradient_checkpointing']:
    model.gradient_checkpointing_enable()
    print("✓ Gradient checkpointing enabled")

print(f"✓ Model initialized on {device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

### Stage 1: Backbone Pretraining
Train the language model on WikiText-103 for next-token prediction.

In [ ]:
print_section_header("Stage 1: Backbone Pretraining")

# Prepare WikiText data
dm = DatasetManager(model.tokenizer_manager, chunk_size=16, num_chunks=4)

token_ids = []
max_tokens = config['datasets']['wikitext']['max_tokens']

print(f"Tokenizing WikiText (max {max_tokens} tokens)...")
for item in wikitext:
    text = item['text']
    if text.strip():
        tokens = model.tokenizer_manager.encode(text)
        token_ids.extend(tokens)
        if len(token_ids) >= max_tokens:
            break

token_ids = token_ids[:max_tokens]
print(f"✓ Total tokens: {len(token_ids)}")

# Create dataloader
batch_size = config['stages']['stage1_backbone']['batch_size']
lm_loader = dm.build_dataloader(token_ids, batch_size=batch_size, shuffle=True)
print(f"✓ Dataloader created (batch_size={batch_size})")

In [ ]:
# Train Stage 1
epochs = config['stages']['stage1_backbone']['epochs']
lr = config['stages']['stage1_backbone']['learning_rate']
save_every = config['stages']['stage1_backbone']['save_every']

print(f"Training for {epochs} epochs (learning rate: {lr})")
print("This may take 30-60 minutes...\n")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=config['training']['weight_decay'])

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    model.train()
    total_loss = 0
    
    for step, batch in enumerate(lm_loader):
        batch = batch.to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            with autocast():
                loss = model.backbone.compute_loss(batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['training']['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = model.backbone.compute_loss(batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['training']['max_grad_norm'])
            optimizer.step()
        
        total_loss += loss.item()
        
        if step % 50 == 0:
            print(f"  Step {step}: Loss = {loss.item():.4f}")
    
    avg_loss = total_loss / len(lm_loader)
    print(f"  Average loss: {avg_loss:.4f}")
    
    # Save checkpoint
    if (epoch + 1) % save_every == 0:
        colab_manager.save_checkpoint(model, f"stage1_epoch{epoch+1}", optimizer, epoch)

# Final checkpoint
colab_manager.save_checkpoint(model, "stage1_final", optimizer, epochs)
print("\n✓ Stage 1 complete!")

### Stage 2: Retrieval Memory Setup
Set up the vector memory with WikiText documents.

In [ ]:
print_section_header("Stage 2: Retrieval Memory Setup")

# Create knowledge ingestion pipeline
pipeline = KnowledgeIngestionPipeline(
    embedding_manager=model.embedding_manager,
    vector_memory=model.memory_manager.semantic,
    state_db_path=config['state_db_path'],
    chunk_size_char=config['stages']['stage2_retrieval']['chunk_size'],
    chunk_overlap_char=config['stages']['stage2_retrieval']['chunk_overlap']
)

# Seed with documents
num_docs = config['stages']['stage2_retrieval']['num_documents']
print(f"Adding {num_docs} documents to memory...")

for i in range(min(num_docs, len(wikitext))):
    doc_id = f"doc_{i}"
    text = wikitext[i]['text']
    if text.strip():
        emb = model.embedding_manager.embed_query(text)
        model.memory_manager.semantic.add_document(doc_id, text, emb, {"source": "wikitext"})

print(f"✓ Memory seeded with {model.memory_manager.semantic.index.ntotal} vectors")

# Save vector memory
os.makedirs(config['vector_memory_dir'], exist_ok=True)
model.memory_manager.semantic.save(config['vector_memory_dir'])
print(f"✓ Vector memory saved to {config['vector_memory_dir']}")

### Stage 3: Planner Training
Train the planner on GSM8K math reasoning problems.

In [ ]:
print_section_header("Stage 3: Planner Training")

# Prepare planner data
planner_questions = [item['question'] for item in gsm8k]
action_types = config['stages']['stage3_planner']['action_types']

# Create dummy action labels (in real training, extract from reasoning)
planner_actions = [i % action_types for i in range(len(planner_questions))]

print(f"Planner dataset: {len(planner_questions)} questions")

# Create dataset and dataloader
planner_dataset = PlannerDataset(planner_questions, planner_actions, model)
batch_size = config['stages']['stage3_planner']['batch_size']
planner_loader = DataLoader(planner_dataset, batch_size=batch_size, shuffle=True)
print(f"✓ Planner dataloader created (batch_size={batch_size})")

In [ ]:
# Train Stage 3
epochs = config['stages']['stage3_planner']['epochs']
lr = config['stages']['stage3_planner']['learning_rate']

print(f"Training for {epochs} epochs (learning rate: {lr})")
print("This may take 20-40 minutes...\n")

planner_optimizer = torch.optim.AdamW(model.planner.parameters(), lr=lr)

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for step, (questions, actions) in enumerate(planner_loader):
        questions = [q.to(device) if hasattr(q, 'to') else q for q in questions]
        actions = actions.to(device)
        
        planner_optimizer.zero_grad()
        
        # Forward pass
        logits = model.planner(questions)
        loss = torch.nn.functional.cross_entropy(logits, actions)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.planner.parameters(), config['training']['max_grad_norm'])
        planner_optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        predictions = logits.argmax(dim=-1)
        correct += (predictions == actions).sum().item()
        total += actions.size(0)
        
        if step % 20 == 0:
            print(f"  Step {step}: Loss = {loss.item():.4f}, Acc = {100*correct/total:.1f}%")
    
    avg_loss = total_loss / len(planner_loader)
    accuracy = 100 * correct / total
    print(f"  Average loss: {avg_loss:.4f}, Accuracy: {accuracy:.1f}%")

# Save checkpoint
colab_manager.save_checkpoint(model, "stage3_final", planner_optimizer, epochs)
print("\n✓ Stage 3 complete!")

### Stage 4: Verifier Training
Train the verifier on fact verification (correct vs incorrect answers).

In [ ]:
print_section_header("Stage 4: Verifier Training")

# Prepare verifier data (correct + incorrect examples)
verifier_texts = []
verifier_labels = []

# Add correct examples
for item in gsm8k:
    verifier_texts.append(item['question'] + " " + item['answer'])
    verifier_labels.append(1)

# Add incorrect examples (corrupted answers)
for item in gsm8k:
    verifier_texts.append(item['question'] + " INCORRECT: " + item['answer'][:50])
    verifier_labels.append(0)

print(f"Verifier dataset: {len(verifier_texts)} examples ({len(verifier_texts)//2} correct, {len(verifier_texts)//2} incorrect)")

# Create dataset and dataloader
verifier_dataset = VerifierDataset(verifier_texts, verifier_labels, model)
batch_size = config['stages']['stage4_verifier']['batch_size']
verifier_loader = DataLoader(verifier_dataset, batch_size=batch_size, shuffle=True)
print(f"✓ Verifier dataloader created (batch_size={batch_size})")

In [ ]:
# Train Stage 4
epochs = config['stages']['stage4_verifier']['epochs']
lr = config['stages']['stage4_verifier']['learning_rate']

print(f"Training for {epochs} epochs (learning rate: {lr})")
print("This may take 20-40 minutes...\n")

verifier_optimizer = torch.optim.AdamW(model.verifier.parameters(), lr=lr)

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for step, (texts, labels) in enumerate(verifier_loader):
        texts = [t.to(device) if hasattr(t, 'to') else t for t in texts]
        labels = labels.to(device)
        
        verifier_optimizer.zero_grad()
        
        # Forward pass
        logits = model.verifier(texts)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.squeeze(), labels.float())
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.verifier.parameters(), config['training']['max_grad_norm'])
        verifier_optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        predictions = (logits.squeeze() > 0).long()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        if step % 20 == 0:
            print(f"  Step {step}: Loss = {loss.item():.4f}, Acc = {100*correct/total:.1f}%")
    
    avg_loss = total_loss / len(verifier_loader)
    accuracy = 100 * correct / total
    print(f"  Average loss: {avg_loss:.4f}, Accuracy: {accuracy:.1f}%")

# Save checkpoint
colab_manager.save_checkpoint(model, "stage4_final", verifier_optimizer, epochs)
print("\n✓ Stage 4 complete!")

### Stage 5: Joint Fine-Tuning
Fine-tune all components together.

In [ ]:
print_section_header("Stage 5: Joint Fine-Tuning")

epochs = config['stages']['stage5_joint']['epochs']
lr = config['stages']['stage5_joint']['learning_rate']

print(f"Training for {epochs} epochs (learning rate: {lr})")
print("This may take 15-30 minutes...\n")

joint_optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    model.train()
    total_loss = 0
    
    # Alternate between tasks
    for step, ((lm_batch, _), (planner_batch, _), (verifier_batch, _)) in enumerate(zip(lm_loader, planner_loader, verifier_loader)):
        joint_optimizer.zero_grad()
        
        # Language modeling loss
        lm_batch = lm_batch.to(device)
        lm_loss = model.backbone.compute_loss(lm_batch)
        
        # Planner loss
        planner_questions = [q.to(device) if hasattr(q, 'to') else q for q in planner_batch[0]]
        planner_actions = planner_batch[1].to(device)
        planner_logits = model.planner(planner_questions)
        planner_loss = torch.nn.functional.cross_entropy(planner_logits, planner_actions)
        
        # Verifier loss
        verifier_texts = [t.to(device) if hasattr(t, 'to') else t for t in verifier_batch[0]]
        verifier_labels = verifier_batch[1].to(device)
        verifier_logits = model.verifier(verifier_texts)
        verifier_loss = torch.nn.functional.binary_cross_entropy_with_logits(verifier_logits.squeeze(), verifier_labels.float())
        
        # Combined loss
        loss = lm_loss + 0.5 * planner_loss + 0.5 * verifier_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['training']['max_grad_norm'])
        joint_optimizer.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            print(f"  Step {step}: Loss = {loss.item():.4f} (LM: {lm_loss.item():.4f}, Planner: {planner_loss.item():.4f}, Verifier: {verifier_loss.item():.4f})")
    
    avg_loss = total_loss / len(lm_loader)
    print(f"  Average loss: {avg_loss:.4f}")

# Save final checkpoint
colab_manager.save_checkpoint(model, "stage5_final", joint_optimizer, epochs)
print("\n✓ Stage 5 complete!")

### Save Final Model
Save the trained model to Google Drive.

In [ ]:
print_section_header("Saving Final Model")

# Save complete model
final_model_path = "/content/harmony_trained.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
    'training_stages': ['stage1', 'stage2', 'stage3', 'stage4', 'stage5']
}, final_model_path)

print(f"✓ Model saved locally: {final_model_path}")

# Copy to Google Drive
import shutil
drive_path = "/content/drive/MyDrive/HARMONY/harmony_trained.pt"
shutil.copy2(final_model_path, drive_path)
print(f"✓ Model saved to Google Drive: {drive_path}")

In [ ]:
# Final summary
print_section_header("Training Complete!")

print("✓ All 5 training stages completed successfully")
print("\nModel saved to:")
print(f"  - Local: {final_model_path}")
print(f"  - Google Drive: {drive_path}")

print("\nCheckpoints saved to:")
print(f"  - Local: {config['checkpoint_dir']}")
print(f"  - Google Drive: /content/drive/MyDrive/HARMONY/")

colab_manager.print_memory_status()

print("\nYou can now:")
print("1. Download the model from Google Drive")
print("2. Use the model for inference")
print("3. Continue training by loading checkpoints")